In [ ]:
import pandas as pd
from groq import Groq
import time

# Initializing the Groq model
API_KEY = "Your api key!"
client = Groq(api_key=API_KEY)

# Loading the dataset
df = pd.read_csv('customer_support_tickets.csv')

print(f"Total tickets loaded: {len(df)}")

Total tickets loaded: 8469


In [18]:
def analyze_ticket(ticket_text):
    """Sends a single ticket to Groq and strictly forces a clean output format."""
    
    prompt = f"""
    You are an enterprise system parser. You must read the customer support ticket text below and output exactly two values separated by a pipe symbol (|):
    
    Format to follow: Churn_Risk_Score | Core_Issue
    
    RULES:
    1. Churn_Risk_Score must be a single digit from 1 (Happy) to 5 (Extremely Angry/Flight Risk).
    2. Core_Issue must be EXACTLY one of these specific words: [Billing, Hardware, Software Bug, User Error, Shipping, Other].
    3. Do NOT include markdown formatting, backticks, bold text, introduction text, or explanations. 
    4. Output absolutely NOTHING else except the 'Score | Category' string.
    
    Ticket Text to Analyze:
    "{ticket_text}"
    """
    
    try:
        chat_completion = client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": prompt,
                }
            ],
            model="llama-3.1-8b-instant", 
            temperature=0.0, # Dropping temperature to 0.0 makes it strictly follow rules
        )
        return chat_completion.choices[0].message.content.strip()
    except Exception as e:
        return f"Error | {str(e)}"

In [ ]:
# Grabbing the first 5 rows
test_df = df.head(5).copy()

print("Analyzing tickets with Groq (Llama 3)...\n")

results = []

for index, row in test_df.iterrows():
    print(f"Processing Ticket ID: {row['Ticket ID']}...")
    
    # Call the API
    ai_result = analyze_ticket(row['Ticket Description'])
    results.append(ai_result)
    
    print(f"AI Output: {ai_result}")
    print("-" * 50)
    
    # Pausing for 2 seconds to respect free-tier rate limits
    time.sleep(2) 

test_df['AI_Analysis'] = results
print("Test Complete!")

Analyzing tickets with Groq (Llama 3)...

Processing Ticket ID: 1...
AI Output: 1 | Billing
--------------------------------------------------
Processing Ticket ID: 2...
AI Output: 1 | Other
--------------------------------------------------
Processing Ticket ID: 3...
AI Output: 2 | Hardware
--------------------------------------------------
Processing Ticket ID: 4...
AI Output: 1 | Other
--------------------------------------------------
Processing Ticket ID: 5...
AI Output: 1 | Other
--------------------------------------------------
Test Complete!


In [ ]:
import time

# 1. Grabbing the first 200 rows
final_df = df.head(200).copy()
print("Starting batch processing of 200 tickets cleanly...\n")

results = []

# 2. Running the loop
for index, row in final_df.iterrows():
    if index % 10 == 0:
        print(f"Processing row {index}/200...")
        
    ai_result = analyze_ticket(row['Ticket Description'])
    results.append(ai_result)
    time.sleep(2) # Speed limit

# 3.  Breaking the strings down safely
churn_scores = []
core_issues = []

for idx, res in enumerate(results):
    # Checking if the output looks correct (contains a pipe symbol '|')
    if "|" in str(res):
        parts = res.split("|")
        # Ensure we got exactly two pieces out of it
        if len(parts) == 2:
            churn_scores.append(parts[0].strip())
            core_issues.append(parts[1].strip())
            continue
            
    # Fallback if the AI returned a broken string or an error code
    churn_scores.append("3")  # Assigning a neutral medium score
    core_issues.append("Other") # Assigning the generic category

# 4. Mapping the safe arrays directly to our new columns
final_df['Churn_Risk_Score'] = churn_scores
final_df['Core_Issue'] = core_issues

# 5. Saving to the new CSV file ready for the database!
export_filename = 'ai_enriched_tickets.csv'
final_df.to_csv(export_filename, index=False)

print(f"\nSuccess! Data safely exported to {export_filename}")

Starting batch processing of 200 tickets cleanly...

Processing row 0/200...
Processing row 10/200...
Processing row 20/200...
Processing row 30/200...
Processing row 40/200...
Processing row 50/200...
Processing row 60/200...
Processing row 70/200...
Processing row 80/200...
Processing row 90/200...
Processing row 100/200...
Processing row 110/200...
Processing row 120/200...
Processing row 130/200...
Processing row 140/200...
Processing row 150/200...
Processing row 160/200...
Processing row 170/200...
Processing row 180/200...
Processing row 190/200...

Success! Data safely exported to ai_enriched_tickets.csv
